# Visualización estadística y espacial de Boyacá

**Propósito:** aprender Matplotlib, su interfaz `pyplot`, Seaborn y GeoPandas con datos oficiales de los municipios de Boyacá. Ejecute las celdas en orden. Cada grupo puede adaptar los patrones al [proyecto final](../../proyecto/Proyecto_final_Boyaca.md).

Los datos incluidos son recortes preparados para el aprendizaje: proyecciones DANE 2020–2025, geometrías MGN 2025 simplificadas y EVA 2024 para papa. Lea [procedencia y límites](../../proyecto/datos/README.md). Las proyecciones son estimaciones, no conteos observados. Los mapas MGN representan límites geoestadísticos, no necesariamente límites legales.

## 1 Preparación y lectura

Use el entorno del curso y ejecute `pip install -r kit/requirements_visualizacion.txt` desde la raíz. Inicie Jupyter con ese entorno; abra este notebook desde el repositorio. Las salidas se guardan en `kit/salidas/visualizacion/`. Los datos preparados van con el repositorio y no requieren red.

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import geopandas as gpd
from scipy.stats import spearmanr
from IPython.display import display

# Localiza la raíz desde kit/Notebooks, kit/ o la raíz del repositorio.
actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents] if (p / "proyecto/datos/poblacion_boyaca_dane_2020_2025.csv").exists()), None)
if raiz is None:
    raise FileNotFoundError("Abra el notebook desde el repositorio; falta proyecto/datos.")
datos = raiz / "proyecto" / "datos"
salidas = raiz / "kit" / "salidas" / "visualizacion"
salidas.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
ANIO = 2024
print("Repositorio:", raiz, "| Año de ejemplo:", ANIO)

In [ ]:
poblacion = pd.read_csv(datos / "poblacion_boyaca_dane_2020_2025.csv", dtype={"codigo_municipio": "string"})
municipios = gpd.read_file(datos / "municipios_boyaca_mgn_2025.geojson")
eva = pd.read_csv(datos / "eva_papa_boyaca_2024.csv", dtype={"codigo_municipio": "string"})
assert not poblacion.duplicated(["codigo_municipio", "anio"]).any()
assert not municipios["codigo_municipio"].duplicated().any()
assert (poblacion["hombres"] + poblacion["mujeres"] == poblacion["poblacion_total"]).all()
assert poblacion["pob_18_44"].le(poblacion["poblacion_total"]).all()
base = municipios.merge(poblacion.query("anio == @ANIO"), on="codigo_municipio", how="left", validate="one_to_one")
base["porcentaje_18_44"] = 100 * base["pob_18_44"] / base["poblacion_total"].where(base["poblacion_total"] > 0)
print("Municipios cartográficos:", len(base), "| sin población:", base.poblacion_total.isna().sum())
display(base.drop(columns="geometry").head())

## 2 Matplotlib y pyplot: diez municipios más poblados

`pyplot` es la interfaz de Matplotlib para crear figuras; `fig, ax` permite controlar ejes, títulos y etiquetas. Una barra parte de cero y compara cantidades en el mismo año. Los 10 municipios se seleccionan por población, no por extensión.

In [ ]:
top_pob = base.nlargest(10, "poblacion_total").sort_values("poblacion_total")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_pob["municipio"], top_pob["poblacion_total"], color="#3478a3")
ax.set(title=f"Boyacá: 10 municipios con mayor población proyectada, {ANIO}", xlabel="Habitantes proyectados", ylabel="Municipio")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.tight_layout()
fig.savefig(salidas / "top_10_poblacion.png", dpi=180, bbox_inches="tight")
plt.show()

## 3 Seaborn: hombres y mujeres en los municipios más extensos

Se seleccionan los 10 polígonos de mayor área calculada antes de simplificar la geometría. Las barras agrupadas muestran las dos series demográficas para esos municipios. El área está en km²; las barras representan población, no extensión ni fuerza laboral.

In [ ]:
top_area = base.nlargest(10, "area_km2").copy()
orden_area = top_area.sort_values("area_km2", ascending=False)["municipio"].tolist()
series = top_area.melt(id_vars=["municipio", "area_km2"], value_vars=["hombres", "mujeres"], var_name="sexo", value_name="poblacion")
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=series, y="municipio", x="poblacion", hue="sexo", order=orden_area, ax=ax, errorbar=None)
ax.set(title=f"Población por sexo en los 10 municipios más extensos, {ANIO}", xlabel="Habitantes proyectados", ylabel="Municipio, ordenado por área")
ax.legend(title="Sexo registrado")
fig.tight_layout(); plt.show()
display(top_area[["municipio", "area_km2", "hombres", "mujeres"]].round({"area_km2": 1}))

## 4 Series temporales para los diez municipios más extensos

Cada panel representa un municipio; cada línea corresponde a hombres o mujeres. El orden por extensión usa MGN 2025 para todos los años; las poblaciones son proyecciones para cada vigencia. Revise la escala del eje antes de comparar cambios.

In [ ]:
serie_tiempo = poblacion.merge(top_area[["codigo_municipio", "area_km2"]], on="codigo_municipio", validate="many_to_one")
serie_tiempo = serie_tiempo.melt(id_vars=["municipio", "anio"], value_vars=["hombres", "mujeres"], var_name="sexo", value_name="poblacion")
g = sns.relplot(data=serie_tiempo, x="anio", y="poblacion", hue="sexo", col="municipio", col_order=orden_area, col_wrap=2, kind="line", marker="o", facet_kws={"sharey": False}, height=2.5, aspect=1.5)
g.set_axis_labels("Año", "Población proyectada")
g.figure.suptitle("Población por sexo en los municipios más extensos", y=1.01)
plt.show()

## 5 Histogramas y diagramas de caja

Un histograma muestra la distribución de una variable municipal. Una caja resume mediana, cuartiles y valores atípicos potenciales; un punto extremo no implica un error. Cada punto es un municipio, no una persona.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(data=base, x="porcentaje_18_44", bins=12, ax=ejes[0], color="#388b78")
ejes[0].set(title=f"Distribución de población de 18 a 44 años, {ANIO}", xlabel="Población de 18 a 44 años (%)", ylabel="Número de municipios")
sns.boxplot(data=base, x="porcentaje_18_44", ax=ejes[1], color="#93c5bb")
ejes[1].set(title="Resumen de la distribución municipal", xlabel="Población de 18 a 44 años (%)")
fig.tight_layout(); plt.show()

## 6 Dispersión y correlación exploratoria

Ejemplo con papa: unimos por DIVIPOLA, año y municipio, y agrupamos los períodos A y B de la categoría **Papa todas las variedades, en fresco**. Rendimiento anual = suma de toneladas / suma de hectáreas cosechadas. La comparación muestra relación entre dos indicadores municipales y no prueba causalidad. No es la prueba de la hipótesis de desempeño; para eso se debe incorporar el puntaje oficial compatible.

In [ ]:
papa = eva.query("anio == 2024").groupby("codigo_municipio", as_index=False).agg(produccion_t=("produccion_t", "sum"), area_cosechada_ha=("area_cosechada_ha", "sum"))
papa["rendimiento_t_ha"] = papa["produccion_t"] / papa["area_cosechada_ha"].where(papa["area_cosechada_ha"] > 0)
base_papa = municipios.merge(poblacion.query("anio == 2024"), on="codigo_municipio", how="left", validate="one_to_one")
base_papa["porcentaje_18_44"] = 100 * base_papa["pob_18_44"] / base_papa["poblacion_total"].where(base_papa["poblacion_total"] > 0)
analisis = base_papa.merge(papa, on="codigo_municipio", how="left", validate="one_to_one")
validos = analisis.dropna(subset=["porcentaje_18_44", "rendimiento_t_ha"])
print(f"Municipios con datos comparables: {len(validos)} de {len(base_papa)}")
if len(validos) >= 3:
    rho, p_valor = spearmanr(validos["porcentaje_18_44"], validos["rendimiento_t_ha"])
    print(f"Spearman: {rho:.3f}; p: {p_valor:.3g}. Análisis exploratorio; no indica causalidad.")
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=validos, x="porcentaje_18_44", y="rendimiento_t_ha", ax=ax, alpha=.75)
ax.set(title="Población joven y rendimiento de papa, 2024", xlabel="Población de 18 a 44 años (%)", ylabel="Rendimiento de papa (t/ha)")
fig.tight_layout(); plt.show()

## 7 Mapas coropléticos municipales

El color representa el indicador de cada municipio; gris significa dato no disponible. El mapa de población es adecuado para cantidades totales; el porcentaje permite comparar la estructura de municipios de distinto tamaño. Se muestran fuentes y vigencias en el pie.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 8))
base.plot(column="poblacion_total", cmap="Blues", legend=True, ax=ejes[0], edgecolor="white", linewidth=.15, missing_kwds={"color": "lightgrey", "label": "Sin datos"})
base.plot(column="porcentaje_18_44", cmap="YlGn", legend=True, ax=ejes[1], edgecolor="white", linewidth=.15, missing_kwds={"color": "lightgrey", "label": "Sin datos"})
ejes[0].set_title(f"Población proyectada, {ANIO}")
ejes[1].set_title(f"Población de 18 a 44 años (%), {ANIO}")
for ax in ejes: ax.set_axis_off()
fig.text(.1, .02, "Fuentes: DANE, proyecciones 2025 y MGN 2025; geometrías simplificadas.", fontsize=9)
fig.tight_layout(rect=(0, .04, 1, 1))
fig.savefig(salidas / "mapas_poblacion.png", dpi=180, bbox_inches="tight")
plt.show()

## 8 Extensión al proyecto final

El conjunto de ejemplo no incluye desempeño municipal ni la capa de capacidad de uso. Para evaluar la hipótesis, incorpore los datos oficiales, valide claves y vigencias, muestre faltantes y compare población joven con el puntaje. Para suelo, use geometrías oficiales de capacidad de uso y una intersección por área; no interprete el rendimiento agrícola como calidad del suelo.

**Ejercicios:** cambie el año; grafique proporción de mujeres jóvenes; compare población total y densidad calculada por área; mapee el rendimiento de papa con datos ausentes; y construya una gráfica de dispersión entre población joven y desempeño usando el conjunto validado por su grupo. Explique la unidad de análisis y las limitaciones en cada caso.